In [1]:
import dashscope
print(dashscope.__version__)

1.27.1


In [2]:
# 导入库
import dashscope
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix


# 设置你的api_key


# 读取数据集
df = pd.read_csv("pima-indians-diabetes.csv")
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# 划分数据集，训练逻辑回归模型
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 随便取一位测试集患者的数据
patient = X_test.iloc[0].to_dict()
risk_prob = model.predict_proba(X_test.iloc[0:1])[0,1]

print("患者指标：", patient)
print(f"模型预测糖尿病患病概率：{risk_prob:.2%}")

# 调用Qwen‑Max给出专业解读
prompt = f"""
你是一名内分泌临床医师。根据下面患者体检指标和模型给出的患病风险，给一段通俗易懂的解读和生活干预建议，不要过度诊断，提示仅供参考，不能替代临床就诊。
患者指标：{patient}
模型预测患病概率：{risk_prob:.2%}
"""

resp = dashscope.Generation.call(
    model="qwen-max",
    messages=[{"role":"user","content": prompt}]
)
if resp.status_code == 200:
    print("\n===== 模型解读 =====")
    print(resp.output.text)
else:
    print("调用失败：",resp.message)


患者指标： {'0': 6.0, '1': 98.0, '2': 58.0, '3': 33.0, '4': 190.0, '5': 34.0, '6': 0.43, '7': 43.0}
模型预测糖尿病患病概率：26.65%

===== 模型解读 =====
根据您提供的体检指标，我来尝试解读一下，并给出一些生活干预的建议。不过，请记得这些建议不能代替专业医生的意见，如果对健康状况有任何疑问，还是应该去医院进行专业的咨询和检查。

从给定的数据来看（假设这些数据依次代表的是：空腹血糖、收缩压、舒张压、腰围、总胆固醇、高密度脂蛋白胆固醇、甘油三酯、年龄），我们可以看出几个可能需要注意的地方：

- 空腹血糖6.0 mmol/L处于正常范围的上限附近（正常值<6.1mmol/L），但接近于糖尿病前期的标准（≥6.1且<7.0mmol/L）。
- 收缩压98mmHg和舒张压58mmHg看起来偏低，但是这个也需要结合个人的具体情况来看待。
- 腰围33英寸（约等于84厘米），对于亚洲人来说属于正常范围内，但仍需注意保持健康的体型。
- 总胆固醇190mg/dL（约等于4.9mmol/L）在理想水平之内。
- 高密度脂蛋白(HDL)胆固醇34mg/dL（约等于0.88mmol/L），略低于推荐水平（男性>40mg/dL, 女性>50mg/dL）。
- 甘油三酯0.43mmol/L远低于标准（<1.7mmol/L）是很好的状态。
- 年龄43岁。

基于模型预测出26.65%的患病风险，虽然不是非常高，但也提示存在一定的健康隐患。这里有几个生活上的小建议可以帮助改善您的健康状况：

1. **均衡饮食**：多吃蔬菜水果，减少加工食品摄入；控制糖分和饱和脂肪酸的摄入量。
2. **规律运动**：每周至少150分钟中等强度或75分钟高强度的身体活动。
3. **保持健康体重**：通过合理膳食与适量运动维持适宜体重。
4. **定期监测血压及血糖**：特别是如果有家族史的话，更加需要关注。
5. **戒烟限酒**：烟草和酒精都是心血管疾病的危险因素。
6. **充足睡眠**：保证每晚7-9小时高质量睡眠。

最后再次强调，以上信息仅供参考，具体情况请遵循专业医护人员指导。希望我的回答对你有所帮助！


In [3]:

from sklearn.preprocessing import StandardScaler

# 特征标准化（对树模型不是必须，但方便和逻辑回归公平对比）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [4]:
# 训练随机森林模型
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
y_pred = model.predict(X_test)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

print(f"逻辑回归准确率：{accuracy_score(y_test,y_pred):.2%}")
print(f"随机森林准确率：{acc_rf:.2%}")


逻辑回归准确率：73.59%
随机森林准确率：75.32%


In [5]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

print("=" * 50)
print("随机森林模型评估结果")
print("=" * 50)

rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

print(f"准确率 (Accuracy):  75.32%")
print(f"精确率 (Precision): {rf_precision:.4f} ({rf_precision*100:.2f}%)")
print(f"召回率 (Recall):    {rf_recall:.4f} ({rf_recall*100:.2f}%)")
print(f"F1分数 (F1-Score):  {rf_f1:.4f} ({rf_f1*100:.2f}%)")

print("\n详细分类报告:")
print(classification_report(y_test, y_pred_rf, target_names=["无糖尿病", "有糖尿病"]))


随机森林模型评估结果
准确率 (Accuracy):  75.32%
精确率 (Precision): 0.6386 (63.86%)
召回率 (Recall):    0.6625 (66.25%)
F1分数 (F1-Score):  0.6503 (65.03%)

详细分类报告:
              precision    recall  f1-score   support

        无糖尿病       0.82      0.80      0.81       151
        有糖尿病       0.64      0.66      0.65        80

    accuracy                           0.75       231
   macro avg       0.73      0.73      0.73       231
weighted avg       0.76      0.75      0.75       231



In [6]:
# 输出混淆矩阵对比
print("=====逻辑回归混淆矩阵=====")
print(confusion_matrix(y_test,y_pred))
print("=====随机森林混淆矩阵=====")
print(confusion_matrix(y_test, y_pred_rf))


=====逻辑回归混淆矩阵=====
[[120  31]
 [ 30  50]]
=====随机森林混淆矩阵=====
[[121  30]
 [ 27  53]]


In [7]:
# 查看特征重要性，知道哪些指标对糖尿病风险影响最大
feature_names = df.columns[:-1]
importances = rf.feature_importances_

for name, score in zip(feature_names, importances):
    print(f"{name}: {score:.3f}")


0: 0.081
1: 0.282
2: 0.084
3: 0.071
4: 0.069
5: 0.158
6: 0.113
7: 0.142


In [8]:
# 拿随机森林给出的患病概率，送给大模型解读
risk_prob_rf = rf.predict_proba(X_test.iloc[0:1])[0,1]

prompt = f"""
你是一名内分泌临床医师。根据下面患者体检指标与随机森林模型给出的患病风险，给出一段通俗易懂的解读和生活干预建议，不要过度诊断，提示仅供参考，不能替代临床诊断。
患者指标：{X_test.iloc[0].to_dict()}
模型预测糖尿病患病概率：{risk_prob_rf:.2%}
"""

resp = dashscope.Generation.call(
    model="qwen-max",
    messages=[{"role":"user","content": prompt}]
)

print("\n=====随机森林模型解读=====")
print(resp.output.text)



=====随机森林模型解读=====
根据您提供的体检指标以及随机森林模型预测的结果来看，这位患者未来患糖尿病的风险为43%，这表示有一定的患病可能性，但并不是非常高。不过，考虑到预防总是优于治疗的原则，我们可以从以下几个方面入手来降低这一风险：

1. **健康饮食**：尽量减少高糖、高脂肪的食物摄入，比如甜饮料、油炸食品等；增加蔬菜和全谷物的比例，它们富含纤维，有助于控制血糖水平。
2. **定期运动**：每周至少150分钟的中等强度运动（如快步走、游泳或骑自行车）可以帮助提高身体对胰岛素的敏感性，从而更好地管理血糖。
3. **保持健康体重**：如果目前处于超重或肥胖状态，减轻体重哪怕只有5%到7%，也能显著降低发展成2型糖尿病的风险。
4. **规律作息**：保证充足睡眠，避免熬夜，因为不规律的生活习惯可能会影响体内激素水平，进而影响血糖调控。
5. **戒烟限酒**：烟草中的有害物质会损害血管，而过量饮酒则可能导致肝脏问题，两者都与糖尿病的发展有关联。

最后，请记得这些只是一般性的建议，并不能代替专业医生的意见。如果担心自己的健康状况，最好咨询专业的医疗人员进行详细的检查和指导。希望以上信息能帮助到您！
